# Exercise 5 — BotRunner end-to-end

`BotRunner` is the complete daily bot: run once, log the result, alert on trades. It tracks how many times it has run and exposes `read_log()` so you can inspect the full session history. This exercise runs two strategies and compares their log entries.

In [ ]:
import pandas as pd, math, datetime, pathlib, tempfile

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
from dataclasses import dataclass, field

@dataclass
class Trade:
    date: object; action: str; price: float
    shares: float; cash_after: float; value_after: float

@dataclass
class PaperAccount:
    initial_cash: float = 10_000.0
    cash:   float = field(init=False)
    shares: float = field(init=False)
    trades: list  = field(init=False)
    def __post_init__(self):
        self.cash = self.initial_cash; self.shares = 0.0; self.trades = []
    def portfolio_value(self, price):
        return self.cash + self.shares * float(price)
    def buy(self, date, price, fraction=1.0):
        price = float(price)
        if self.cash <= 0 or price <= 0: return None
        shares = (self.cash * fraction) / price
        cost   = shares * price
        if cost > self.cash: shares = self.cash / price; cost = shares * price
        self.cash -= cost; self.shares += shares
        t = Trade(date=date, action="BUY", price=price, shares=shares,
                  cash_after=self.cash, value_after=self.portfolio_value(price))
        self.trades.append(t); return t
    def sell(self, date, price):
        price = float(price)
        if self.shares <= 0: return None
        proceeds = self.shares * price; sold = self.shares
        self.cash += proceeds; self.shares = 0.0
        t = Trade(date=date, action="SELL", price=price, shares=sold,
                  cash_after=self.cash, value_after=self.portfolio_value(price))
        self.trades.append(t); return t

def run_paper_trader(df, signals, initial_cash=10_000.0, fraction=1.0):
    acc = PaperAccount(initial_cash=initial_cash)
    eq  = []; prev = 0
    for i in range(len(df)):
        date = df.index[i]; price = float(df["Close"].iloc[i])
        sig = int(signals.iloc[i])
        if sig == 1 and prev == 0: acc.buy(date, price, fraction=fraction)
        elif sig == 0 and prev == 1: acc.sell(date, price)
        eq.append(acc.portfolio_value(price)); prev = sig
    if acc.shares > 0: acc.sell(df.index[-1], float(df["Close"].iloc[-1]))
    equity = pd.Series(eq, index=df.index)
    tr     = float(equity.iloc[-1] / initial_cash - 1.0)
    peak   = equity.cummax()
    return {"account": acc, "trades": acc.trades, "equity": equity,
            "initial_cash": initial_cash, "final_value": float(equity.iloc[-1]),
            "total_return": tr, "max_drawdown": float(((equity - peak)/peak).min()),
            "n_trades": len(acc.trades),
            "n_buys":   sum(1 for t in acc.trades if t.action == "BUY"),
            "n_sells":  sum(1 for t in acc.trades if t.action == "SELL")}

def format_report(result):
    lines = [
        "=== Paper Trading Report ===",
        f"Initial cash :  ${result['initial_cash']:>12,.2f}",
        f"Final value  :  ${result['final_value']:>12,.2f}",
        f"Total return :  {result['total_return']:>12.2%}",
        f"Max drawdown :  {result['max_drawdown']:>12.2%}",
        f"Trades total :  {result['n_trades']:>12d}",
        f"  Buys       :  {result['n_buys']:>12d}",
        f"  Sells      :  {result['n_sells']:>12d}",
    ]
    if result["trades"]:
        f_ = result["trades"][0]; l_ = result["trades"][-1]
        lines.append(f"First trade  :  {f_.action} @ ${f_.price:,.2f}  ({f_.date})")
        lines.append(f"Last trade   :  {l_.action} @ ${l_.price:,.2f}  ({l_.date})")
    return "\n".join(lines)
def format_log_entry(report_text):
    ts  = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    bar = "=" * 52
    return f"\n{bar}\n[{ts}]\n{bar}\n{report_text}"

def log_result(report_text, path):
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    entry = format_log_entry(report_text)
    with open(path, "a", encoding="utf-8") as fh:
        fh.write(entry + "\n")

def send_alert(message, webhook_url=None):
    if webhook_url is None:
        print(f"[ALERT] {message}")
        return True
    try:
        import requests
        resp = requests.post(webhook_url, json={"text": message}, timeout=5)
        return resp.ok
    except Exception:
        return False

def next_run_time(run_time_str="16:00"):
    now = datetime.datetime.now()
    h, m = (int(x) for x in run_time_str.split(":"))
    target = now.replace(hour=h, minute=m, second=0, microsecond=0)
    if target <= now:
        target += datetime.timedelta(days=1)
    return target

def seconds_until(target_dt):
    delta = target_dt - datetime.datetime.now()
    return max(0.0, delta.total_seconds())

def run_daily_loop(bot_fn, run_time="16:00", max_iterations=None,
                   _sleep_fn=None):
    import time
    if _sleep_fn is None: _sleep_fn = time.sleep
    count = 0
    while max_iterations is None or count < max_iterations:
        _sleep_fn(seconds_until(next_run_time(run_time)))
        bot_fn()
        count += 1
    return count

class BotRunner:
    """Stateful daily-bot wrapper."""

    def __init__(self, log_path, webhook_url=None, run_time="16:00"):
        self.log_path    = pathlib.Path(log_path)
        self.webhook_url = webhook_url
        self.run_time    = run_time
        self._run_count  = 0

    def run_once(self, df, signals, initial_cash=10_000.0, fraction=1.0):
        """Run the full pipeline: trade → log → alert.

        Steps:
          1. result = run_paper_trader(df, signals, initial_cash, fraction)
          2. report = format_report(result)
          3. log_result(report, self.log_path)
          4. if result["n_trades"] > 0: send_alert(...)
          5. self._run_count += 1
          6. result["report"] = report; return result
        """
        # TODO: ~7 lines
        return {}

    def run_count(self):
        # TODO: return self._run_count
        return 0

    def read_log(self):
        if not self.log_path.exists(): return ""
        return self.log_path.read_text(encoding="utf-8")


### Checks

In [ ]:
checks = 0

# 1 — run_once returns dict with "report" key
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    with tempfile.TemporaryDirectory() as td:
        runner = BotRunner(pathlib.Path(td) / "bot.log")
        result = runner.run_once(df, sig)
    assert isinstance(result, dict)
    assert "report" in result, "result should have 'report' key"
    assert isinstance(result["report"], str) and len(result["report"]) > 20
    checks += 1; print("✅ 1 run_once returns dict with non-empty 'report' key")
except Exception as e:
    print("❌ 1:", e)

# 2 — run_once writes to log file
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    with tempfile.TemporaryDirectory() as td:
        log_p  = pathlib.Path(td) / "bot.log"
        runner = BotRunner(log_p)
        runner.run_once(df, sig)
        assert log_p.exists(), "log file should be created"
        content = log_p.read_text()
        assert "Paper Trading Report" in content
        checks += 1; print("✅ 2 run_once writes to log file")
except Exception as e:
    print("❌ 2:", e)

# 3 — run_count increments each call
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    with tempfile.TemporaryDirectory() as td:
        runner = BotRunner(pathlib.Path(td) / "bot.log")
        assert runner.run_count() == 0
        runner.run_once(df, sig)
        assert runner.run_count() == 1
        runner.run_once(df, sig)
        assert runner.run_count() == 2
    checks += 1; print("✅ 3 run_count increments correctly: 0 → 1 → 2")
except Exception as e:
    print("❌ 3:", e)

# 4 — read_log returns empty string before first run
try:
    with tempfile.TemporaryDirectory() as td:
        runner = BotRunner(pathlib.Path(td) / "bot.log")
        assert runner.read_log() == ""
    checks += 1; print("✅ 4 read_log() returns '' before any run")
except Exception as e:
    print("❌ 4:", e)

# 5 — read_log grows with multiple runs
try:
    df   = _synthetic()
    sig1 = pd.Series(1, index=df.index)
    sig2 = pd.Series(0, index=df.index)
    with tempfile.TemporaryDirectory() as td:
        runner = BotRunner(pathlib.Path(td) / "bot.log")
        runner.run_once(df, sig1)
        runner.run_once(df, sig2)
        log    = runner.read_log()
        n_entries = log.count("=== Paper Trading Report ===")
        assert n_entries == 2, f"expected 2 report entries in log, got {n_entries}"
    checks += 1; print("✅ 5 read_log() has 2 report entries after 2 runs")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
